# Module 30 — Matrix Factorization (SVD) for Collaborative Filtering

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Scipy, Plotly  
> **Prerequisite**: Module 29 (Two-Tower Retrieval)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Rating Dataset](#3-synthetic-rating-dataset)
4. [SVD Implementation](#4-svd-implementation)
5. [ALS Implementation](#5-als-implementation)
6. [Evaluation](#6-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why matrix factorization for recommendations?

Matrix factorization is a **classic** collaborative filtering approach:
- **Latent factors**: Discover hidden user/item characteristics.
- **Sparsity handling**: Work with sparse rating matrices.
- **Scalability**: Efficient for large-scale datasets.

**Applications at Yandex**:
1. **Movie recommendations**: Predict user ratings for movies.
2. **Product recommendations**: Suggest products based on purchase history.
3. **Content personalization**: Customize news feed for users.

**Key algorithms**:
- **SVD**: Singular Value Decomposition of rating matrix.
- **ALS**: Alternating Least Squares optimization.
- **SGD**: Stochastic Gradient Descent for matrix factorization.

In this notebook we will:
1. Generate a synthetic movie rating dataset.
2. Implement SVD-based matrix factorization.
3. Implement ALS optimization.
4. Evaluate with RMSE on held-out ratings.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Matrix factorization ─────────────────────────────────────────
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Rating Dataset

In [ ]:
# Generate synthetic movie rating dataset
N_USERS = 5000
N_MOVIES = 1000
N_RATINGS = 100000

# Generate ratings with some structure
# Users have preferences for certain genres
user_preferences = np.random.randn(N_USERS, 5)  # 5 latent factors
movie_features = np.random.randn(N_MOVIES, 5)   # 5 latent factors

# Generate ratings
ratings = []
for _ in range(N_RATINGS):
    user_id = np.random.randint(0, N_USERS)
    movie_id = np.random.randint(0, N_MOVIES)
    
    # Base rating from latent factors
    base_rating = np.dot(user_preferences[user_id], movie_features[movie_id])
    
    # Add noise and clip to 1-5
    rating = np.clip(base_rating + np.random.normal(0, 0.5), 1, 5)
    rating = round(rating * 2) / 2  # Round to 0.5
    
    ratings.append({
        'user_id': user_id,
        'movie_id': movie_id,
        'rating': rating
    })

df_ratings = pd.DataFrame(ratings)

# Remove duplicates (keep first)
df_ratings = df_ratings.drop_duplicates(subset=['user_id', 'movie_id'], keep='first')

print(f"✓ Generated {len(df_ratings)} ratings")
print(f"  Users: {N_USERS}")
print(f"  Movies: {N_MOVIES}")
print(f"  Sparsity: {1 - len(df_ratings) / (N_USERS * N_MOVIES):.4f}")
print(f"\nRating distribution:")
print(df_ratings['rating'].value_counts().sort_index())

---
## §4 · SVD Implementation

In [ ]:
# Create user-item matrix
user_item_matrix = df_ratings.pivot(
    index='user_id',
    columns='movie_id',
    values='rating'
).fillna(0)

# Convert to sparse matrix
sparse_matrix = csr_matrix(user_item_matrix.values)

# Apply SVD
N_FACTORS = 20
U, sigma, Vt = svds(sparse_matrix, k=N_FACTORS)

# Convert sigma to diagonal matrix
sigma = np.diag(sigma)

print(f"✓ SVD decomposition complete")
print(f"  User matrix U: {U.shape}")
print(f"  Singular values: {sigma.shape}")
print(f"  Item matrix Vt: {Vt.shape}")
print(f"  Latent factors: {N_FACTORS}")

# Reconstruct predictions
predicted_ratings = np.dot(np.dot(U, sigma), Vt)
df_predicted = pd.DataFrame(
    predicted_ratings,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print(f"\n✓ Predicted ratings matrix: {df_predicted.shape}")

---
## §5 · ALS Implementation

In [ ]:
# Implement ALS (Alternating Least Squares)
class ALSRecommender:
    def __init__(self, n_factors=20, n_iterations=10, lambda_reg=0.1):
        self.n_factors = n_factors
        self.n_iterations = n_iterations
        self.lambda_reg = lambda_reg
    
    def fit(self, ratings_matrix):
        n_users, n_items = ratings_matrix.shape
        
        # Initialize latent factors
        self.user_factors = np.random.randn(n_users, self.n_factors) * 0.1
        self.item_factors = np.random.randn(n_items, self.n_factors) * 0.1
        
        # Mask for observed ratings
        mask = ratings_matrix > 0
        
        for iteration in range(self.n_iterations):
            # Update user factors
            for u in range(n_users):
                rated_items = mask[u]
                if rated_items.sum() > 0:
                    A = self.item_factors[rated_items].T @ self.item_factors[rated_items] + \
                        self.lambda_reg * np.eye(self.n_factors)
                    b = self.item_factors[rated_items].T @ ratings_matrix[u, rated_items]
                    self.user_factors[u] = np.linalg.solve(A, b)
            
            # Update item factors
            for i in range(n_items):
                rated_users = mask[:, i]
                if rated_users.sum() > 0:
                    A = self.user_factors[rated_users].T @ self.user_factors[rated_users] + \
                        self.lambda_reg * np.eye(self.n_factors)
                    b = self.user_factors[rated_users].T @ ratings_matrix[rated_users, i]
                    self.item_factors[i] = np.linalg.solve(A, b)
            
            # Compute loss
            predicted = self.user_factors @ self.item_factors.T
            loss = np.sum((mask * (ratings_matrix - predicted)) ** 2)
            
            if (iteration + 1) % 5 == 0:
                print(f"  Iteration {iteration + 1}: Loss = {loss:.4f}")
        
        return self
    
    def predict(self, user_id, item_id):
        return np.dot(self.user_factors[user_id], self.item_factors[item_id])

# Train ALS model
print("Training ALS model...")
als_model = ALSRecommender(n_factors=20, n_iterations=10, lambda_reg=0.1)
als_model.fit(user_item_matrix.values)

print(f"\n✓ ALS model trained")

---
## §6 · Evaluation

In [ ]:
# Evaluate models
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Create test set
test_size = int(len(df_ratings) * 0.2)
df_test = df_ratings.sample(test_size, random_state=42)

# SVD predictions
svd_predictions = []
actuals = []

for _, row in df_test.iterrows():
    user_id = int(row['user_id'])
    movie_id = int(row['movie_id'])
    
    if user_id in df_predicted.index and movie_id in df_predicted.columns:
        pred = df_predicted.loc[user_id, movie_id]
        svd_predictions.append(pred)
        actuals.append(row['rating'])

# Compute metrics
svd_rmse = np.sqrt(mean_squared_error(actuals, svd_predictions))
svd_mae = mean_absolute_error(actuals, svd_predictions)

print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)
print(f"\nSVD Model:")
print(f"  RMSE: {svd_rmse:.4f}")
print(f"  MAE: {svd_mae:.4f}")
print(f"\n💡 Lower RMSE indicates better prediction accuracy")

---
## §7 · Visualization

In [ ]:
# Visualize latent factors
fig = make_subplots(rows=1, cols=2, subplot_titles=['Singular Values', 'User Latent Factors'])

# Singular values
fig.add_trace(go.Scatter(
    x=list(range(len(np.diag(sigma)))),
    y=np.diag(sigma),
    mode='lines+markers',
    name='Singular Values'
), row=1, col=1)

# User latent factors (first 2 dimensions)
fig.add_trace(go.Scatter(
    x=U[:100, 0],
    y=U[:100, 1],
    mode='markers',
    name='Users',
    marker=dict(color='#636EFA', size=6)
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Singular values show importance of each latent factor")
print("   - Users cluster in latent factor space")
print("   - Similar users have similar latent representations")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Recommendation Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Rating prediction, collaborative filtering, cold-start |
> | **Algorithm** | SVD for simplicity, ALS for scalability |
> | **Factors** | 20-100 latent factors depending on data size |
> | **Evaluation** | RMSE, MAE, Precision@K, Recall@K |
> | **Deployment** | Pre-compute factors, update periodically |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Hybrid approach**:
>    ```
>    Collaborative filtering (matrix factorization) + Content-based features
>    ```
>
> 2. **Handling cold-start**:
>    | User Type | Approach |
>    |-----------|----------|
>    | New user | Content-based until enough interactions |
>    | New item | Use item features, popularity baseline |
>    | Existing | Collaborative filtering |
>
> 3. **Production considerations**:
>    - Retrain weekly on fresh interaction data
>    - A/B test against baseline (popularity, random)
>    - Monitor RMSE and business metrics (CTR, conversion)
>
> ### 🔑 Key Takeaways
>
> 1. **Matrix factorization discovers latent factors** that explain user preferences.
> 2. **SVD is simple and effective** for moderate-sized datasets.
> 3. **ALS scales better** for large sparse matrices.
> 4. **20-100 latent factors** typically work well.
> 5. **Combine with content-based methods** for cold-start problems.